In [6]:
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from collections import deque
from video_processing import VideoLipProcessor

class RealTimeLipReader:
    def __init__(self, model_path, vocab, sequence_length=30, frame_size=(100, 50)):
        """Initialize the lip-reading system"""
        self.model = tf.keras.models.load_model(model_path, compile=False)  # Load trained model without compiling
        self.vocab = vocab
        self.sequence_length = sequence_length
        self.frame_size = frame_size
        self.frame_buffer = deque(maxlen=sequence_length)
        self.processor = VideoLipProcessor()

    def run(self):
        """Start real-time lip reading"""
        cap = cv2.VideoCapture(0)  # Open webcam
        if not cap.isOpened():
            print("❌ Error: Could not open webcam")
            return

        print("✅ Webcam opened successfully! Type 'exit' and press Enter to stop.")

        # Instead of `cv2.waitKey()`, use input() to manually exit in Jupyter Notebook
        stop = False

        while not stop:
            ret, frame = cap.read()  # Capture frame
            if not ret:
                print("❌ Error: Failed to read frame")
                break

            processed_frame = self.processor.preprocess_frame(frame)  # Extract lip region

            if processed_frame is not None:
                self.frame_buffer.append(processed_frame)

            # Make a prediction if buffer is full
            if len(self.frame_buffer) == self.sequence_length:
                input_data = np.expand_dims(self.frame_buffer, axis=0)
                prediction = self.model.predict(input_data, verbose=0)
                word = self.vocab[np.argmax(prediction)]
                print(f"🗣️ Predicted Word: {word}")

                # Convert OpenCV BGR to RGB for display
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

                # Use Matplotlib to display the frame (since cv2.imshow doesn't work in Jupyter)
                plt.imshow(frame_rgb)
                plt.title(f"Prediction: {word}")
                plt.axis("off")
                plt.show()

            # Manual exit (instead of cv2.waitKey())
            stop = input("Type 'exit' to stop: ").strip().lower() == 'exit'

        cap.release()
        print("📌 Webcam closed. Exiting program.")

# Example usage
if __name__ == "__main__":
    vocab = ["hello", "bye", "can", "demo", "dog", "here", "lips", "is", "my", "read"]
    reader = RealTimeLipReader("lipreading_model.h5", vocab)
    reader.run()  # Start real-time inference


✅ Webcam opened successfully! Type 'exit' and press Enter to stop.


error: OpenCV(4.11.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\resize.cpp:4208: error: (-215:Assertion failed) !ssize.empty() in function 'cv::resize'
